# AI Lecture Notes Processor - Prototype
Pipeline: **PDF/text -> Summary -> Key concepts -> MCQ quiz**

Run the cells in order with **Shift + Enter**. The first run downloads the AI models (about 2.5 GB), so it needs internet and a few minutes.

## Step 1 - Imports

In [1]:
import re, random, textwrap
import pdfplumber
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from keybert import KeyBERT

print("All libraries loaded")

C:\Users\Nathanal Aji\AppData\Local\Programs\Python\Python315\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries loaded


## Step 2 - Get text (PDF or sample text)

In [2]:
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()

def split_sentences(text):
    text = re.sub(r"\s+", " ", text)
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]

SAMPLE_TEXT = """
Machine learning is a subset of artificial intelligence that enables systems to learn from data
without being explicitly programmed. Supervised learning uses labelled data to train models.
Unsupervised learning finds hidden patterns in unlabelled data. Neural networks are inspired by
the human brain and consist of layers of nodes. Gradient descent is an optimisation algorithm used
to minimise loss functions. Overfitting occurs when a model learns noise in training data and
performs poorly on new data. Cross-validation is a technique used to evaluate how well a model
generalises to unseen data. Reinforcement learning trains an agent to make decisions by rewarding
desirable actions and penalising undesirable ones.
"""

# To use your own PDF, put it in the same folder as this notebook and use:
# text = extract_text_from_pdf("my_lecture.pdf")
text = SAMPLE_TEXT

print(f"Text ready: {len(text.split())} words")

Text ready: 107 words


## Step 3 - Summarisation (BART)

In [6]:
SUM_NAME = "facebook/bart-large-cnn"
print("Loading summarisation model...")
sum_tok = AutoTokenizer.from_pretrained(SUM_NAME)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_NAME)
sum_model.eval()

def _generate(model, tok, prompt, max_in, max_new, min_new=None, beams=4):
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=max_in)
    kwargs = dict(max_new_tokens=max_new, num_beams=beams, early_stopping=True)
    if min_new:
        kwargs["min_new_tokens"] = min_new
    with torch.no_grad():
        out = model.generate(**inputs, **kwargs)
    return tok.decode(out[0], skip_special_tokens=True)

def chunk_text(text, words_per_chunk=450):
    chunks, current, count = [], [], 0
    for s in split_sentences(text):
        n = len(s.split())
        if count + n > words_per_chunk and current:
            chunks.append(" ".join(current))
            current, count = [], 0
        current.append(s)
        count += n
    if current:
        chunks.append(" ".join(current))
    return chunks

def summarise_text(text):
    parts = []
    for chunk in chunk_text(text):
        words = len(chunk.split())
        max_new = 130 if words > 150 else max(30, words // 2)
        min_new = 40 if words > 150 else 10
        parts.append(_generate(sum_model, sum_tok, chunk, 1024, max_new, min_new))
    return " ".join(parts)

summary = summarise_text(text)
print("SUMMARY:")
print(textwrap.fill(summary, 80))

Loading summarisation model...


Loading weights: 100%|██████████| 511/511 [00:02<00:00, 193.87it/s]
[transformers] Both `max_new_tokens` (=53) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SUMMARY:
Machine learning is a subset of artificial intelligence that enables systems to
learn from data without being explicitly programmed. Supervised learning uses
labelled data to train models. Unsupervised learning finds hidden patterns in
unlabelled data.


## Step 4 - Key concept extraction (KeyBERT)

In [7]:
kw_model = KeyBERT()

def extract_keywords(text, top_n=12):
    return kw_model.extract_keywords(
        text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=top_n
    )

keywords = extract_keywords(text)
print("KEY CONCEPTS:")
for kw, score in keywords:
    print(f"  - {kw.title()}  (relevance {score:.2f})")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6753.80it/s]


KEY CONCEPTS:
  - Model Learns  (relevance 0.55)
  - Data Train  (relevance 0.52)
  - Data Neural  (relevance 0.52)
  - Programmed Supervised  (relevance 0.52)
  - Training Data  (relevance 0.52)
  - Supervised  (relevance 0.51)
  - Data Reinforcement  (relevance 0.51)
  - Learn Data  (relevance 0.51)
  - Machine Learning  (relevance 0.51)
  - Reinforcement Learning  (relevance 0.50)
  - Supervised Learning  (relevance 0.50)
  - Learns  (relevance 0.48)


In [9]:
!pip install tiktoken

   ---------------------------------------- 0.0/988.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/988.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/988.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/988.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/988.2 kB ? eta -:--:--
   ---------- ----------------------------- 262.1/988.2 kB ? eta -:--:--
   -------------------- ----------------- 524.3/988.2 kB 964.1 kB/s eta 0:00:01
   ---------------------------------------- 988.2/988.2 kB 1.6 MB/s  0:00:01



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 5 - Multiple-choice quiz generation (T5)
Each key concept becomes the **correct answer**. T5 writes a question about it, and other key concepts act as **distractors**.

In [10]:
# Step 5 - Quiz Generation (simple version)
import random

def generate_quiz_simple(text, keywords, num_questions=5):
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if len(s.strip()) > 30]
    keyword_list = [kw for kw, score in keywords]
    quiz = []
    
    for i in range(min(num_questions, len(sentences))):
        sentence = sentences[i]
        # Find keyword in sentence
        answer = None
        for kw in keyword_list:
            if kw.lower() in sentence.lower():
                answer = kw
                break
        if not answer:
            continue
            
        # Make wrong options
        distractors = [kw for kw in keyword_list if kw != answer][:3]
        if len(distractors) < 3:
            distractors += ["Deep Learning", "Statistics", "Regression"]
        distractors = distractors[:3]
        
        options = distractors + [answer]
        random.shuffle(options)
        correct_letter = ["A","B","C","D"][options.index(answer)]
        
        quiz.append({
            "question": f"Which concept is described: '{sentence[:80]}...'?",
            "options": options,
            "answer": correct_letter
        })
    return quiz

quiz = generate_quiz_simple(text, keywords)

print("QUIZ:")
for i, q in enumerate(quiz):
    print(f"\nQ{i+1}. {q['question']}")
    for letter, opt in zip(["A","B","C","D"], q["options"]):
        print(f"   {letter}) {opt}")
    print(f"   ✅ Answer: {q['answer']}")

QUIZ:

Q1. Which concept is described: 'Machine learning is a subset of artificial intelligence that enables systems to ...'?
   A) data neural
   B) data train
   C) machine learning
   D) model learns
   ✅ Answer: C

Q2. Which concept is described: 'Supervised learning uses labelled data to train models...'?
   A) supervised
   B) data train
   C) model learns
   D) data neural
   ✅ Answer: A

Q3. Which concept is described: 'Unsupervised learning finds hidden patterns in unlabelled data...'?
   A) data neural
   B) model learns
   C) data train
   D) supervised
   ✅ Answer: D


## Step 6 - Full pipeline (one call)

In [24]:
def run_full_pipeline(source, is_pdf=False, num_questions=5):
    content = extract_text_from_pdf(source) if is_pdf else source
    print(f"Processing {len(content.split())} words...\n")

    summary = summarise_text(content)
    kws = extract_keywords(content)
    quiz = generate_mcqs(content, kws, num_questions)

    print("=" * 60 + "\nSUMMARY\n" + "=" * 60)
    print(textwrap.fill(summary, 80))
    print("\n" + "=" * 60 + "\nKEY CONCEPTS\n" + "=" * 60)
    for kw, _ in kws[:8]:
        print(f"  - {kw.title()}")
    print("\n" + "=" * 60 + "\nQUIZ\n" + "=" * 60)
    show_quiz(quiz)
    return {"summary": summary, "keywords": kws, "quiz": quiz}

# Try with text:
result = run_full_pipeline(SAMPLE_TEXT)

# Try with a PDF (uncomment and put your file next to this notebook):
# result = run_full_pipeline("my_lecture.pdf", is_pdf=True)

Processing 107 words...



[transformers] Both `max_new_tokens` (=53) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SUMMARY
Machine learning is a subset of artificial intelligence that enables systems to
learn from data without being explicitly programmed. Supervised learning uses
labelled data to train models. Unsupervised learning finds hidden patterns in
unlabelled data.

KEY CONCEPTS
  - Model Learns
  - Data Train
  - Data Neural
  - Programmed Supervised
  - Training Data
  - Supervised
  - Data Reinforcement
  - Learn Data

QUIZ

Q1. Which concept is described: 'Machine learning is a subset of artificial intelligence that enables systems to ...'?
   A) machine learning
   B) data train
   C) data neural
   D) model learns
   ✅ Answer: A

Q2. Which concept is described: 'Supervised learning uses labelled data to train models...'?
   A) data train
   B) model learns
   C) supervised
   D) data neural
   ✅ Answer: C

Q3. Which concept is described: 'Unsupervised learning finds hidden patterns in unlabelled data...'?
   A) model learns
   B) data neural
   C) supervised
   D) data train
   ✅ Answ

In [ ]:
!pip install --upgrade transformers torch sentencepiece

In [12]:
# Fix - link the two function names together
generate_mcqs = generate_quiz_simple

In [14]:
# Fix - define show_quiz function
def show_quiz(quiz):
    for i, q in enumerate(quiz):
        print(f"\nQ{i+1}. {q['question']}")
        for letter, opt in zip(["A","B","C","D"], q["options"]):
            print(f"   {letter}) {opt}")
        print(f"   ✅ Answer: {q['answer']}")

In [21]:
# Install pandas first
import subprocess
subprocess.run(["pip", "install", "pandas"], check=True)

import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

print("TRAIN columns:", df_train.columns.tolist())
print("TEST columns:", df_test.columns.tolist())
print(f"\nTrain rows: {len(df_train)}")
print(f"Test rows:  {len(df_test)}")
print("\nFirst row of train:")
print(df_train.head(1))

TRAIN columns: ['dialogueID', 'Context', 'QuestionID', 'Question', 'Answer', 'Answer Start', 'Answer End']
TEST columns: ['dialogueID', 'Context', 'QuestionID', 'Question', 'Answer', 'Answer Start', 'Answer End']

Train rows: 29150
Test rows:  7288

First row of train:
  dialogueID                                            Context  QuestionID  \
0   1000.tsv  I'm having a permissions problem with my camco...  1000.tsv_1   

                                           Question                 Answer  \
0  What is the problem with my camcorder on Ubuntu?  a permissions problem   

   Answer Start  Answer End  
0            11          32  


In [22]:
import re

def clean_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return None
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)                  # extra spaces
    text = re.sub(r'http\S+', '', text)                # remove URLs
    text = re.sub(r'[^\w\s\.\,\?\!\:\-]', '', text)   # special chars
    text = re.sub(r'\n+', ' ', text)                   # newlines
    return text if len(text.split()) > 3 else None     # remove too short

# Clean TRAIN
df_train['Context_clean']  = df_train['Context'].apply(clean_text)
df_train['Question_clean'] = df_train['Question'].apply(clean_text)
df_train['Answer_clean']   = df_train['Answer'].apply(clean_text)
df_train_clean = df_train.dropna(
    subset=['Context_clean','Question_clean','Answer_clean'])
df_train_clean = df_train_clean.drop_duplicates(subset=['Question_clean'])

# Clean TEST
df_test['Context_clean']  = df_test['Context'].apply(clean_text)
df_test['Question_clean'] = df_test['Question'].apply(clean_text)
df_test['Answer_clean']   = df_test['Answer'].apply(clean_text)
df_test_clean = df_test.dropna(
    subset=['Context_clean','Question_clean','Answer_clean'])
df_test_clean = df_test_clean.drop_duplicates(subset=['Question_clean'])

print(f"✅ TRAIN: {len(df_train)} → {len(df_train_clean)} rows after cleaning")
print(f"✅ TEST:  {len(df_test)} → {len(df_test_clean)} rows after cleaning")
print("\nSample cleaned row:")
print(df_train_clean[['Context_clean','Question_clean','Answer_clean']].head(2))

✅ TRAIN: 29150 → 4752 rows after cleaning
✅ TEST:  7288 → 1060 rows after cleaning

Sample cleaned row:
                                       Context_clean  \
5  anybody... : plz somone help expert : im a lin...   
6  anybody... : plz somone help expert : im a lin...   

                                  Question_clean  \
5               How did the process work for me?   
6  What was the last stage that didnt get stuck?   

                                        Answer_clean  
5  quit in the user info part, when you set the t...  
6          it was my last stage that didnt get stuck  


In [23]:
# Save cleaned versions
df_train_clean.to_csv("hqa_train_cleaned.csv", index=False)
df_test_clean.to_csv("hqa_test_cleaned.csv", index=False)

print("✅ Saved: hqa_train_cleaned.csv")
print("✅ Saved: hqa_test_cleaned.csv")
print(f"✅ Train rows ready for training: {len(df_train_clean)}")
print(f"✅ Test rows ready for evaluation: {len(df_test_clean)}")

✅ Saved: hqa_train_cleaned.csv
✅ Saved: hqa_test_cleaned.csv
✅ Train rows ready for training: 4752
✅ Test rows ready for evaluation: 1060
